In [3]:
pip install rasterio

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import json
import time
import traceback
import numpy as np
from pathlib import Path
from datetime import datetime

from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import pandas as pd
import rasterio
from rasterio.windows import Window


torch.set_num_threads(1)
torch.set_num_interop_threads(1)


# =========================
# CONFIG YOU MUST SET
# =========================
DATASET_DIR = "/kaggle/input/prometheus"

DATA_ROOT = f"{DATASET_DIR}/data_processed_normalized"
FIRE_ROOT = f"{DATASET_DIR}/fire16"

TRAIN_CSV = f"{DATASET_DIR}/dataset/dataset_index_train_vr50.csv"
VAL_CSV   = f"{DATASET_DIR}/dataset/dataset_index_val.csv"
TEST_CSV  = f"{DATASET_DIR}/dataset/dataset_index_test.csv"

VARS = ["ndvi16", "temp16", "precip16", "rh16", "vpd16", "elevation", "slope"]

PATCH_SIZE = 32
BATCH_SIZE = 32
EPOCHS = 6
LR = 1e-3
THR = 0.5

# Kaggle stability: set 0 by default to avoid worker shutdown spam
NUM_WORKERS = 4

# Save N prediction samples
MAX_SAVE_IMAGES = 24

# Threshold sweep settings
THR_SWEEP = [round(x, 2) for x in np.arange(0.10, 0.91, 0.05)]


# =========================
# RUN OUTPUT DIR
# =========================
RUN_ID = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path(f"/kaggle/working/runs/3dcnn_{RUN_ID}")
RUN_DIR.mkdir(parents=True, exist_ok=True)

CKPT_BEST = RUN_DIR / "best.pt"
CKPT_LAST = RUN_DIR / "last.pt"
LOG_CSV   = RUN_DIR / "train_log.csv"
CFG_JSON  = RUN_DIR / "config.json"
ERR_LOG   = RUN_DIR / "errors.log"

SAMPLES_DIR = RUN_DIR / "pred_samples"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# DATASET
# =========================
class FireConvLSTMDataset(Dataset):
    def __init__(self, index_csv, data_root, fire_root, variables, patch_size=32):
        self.data_root = Path(data_root)
        self.fire_root = Path(fire_root)
        self.variables = variables
        self.patch = patch_size

        time_cols = ["t1", "t2", "t3", "t4"]
        dtype_map = {c: "string" for c in time_cols}
        self.df = pd.read_csv(index_csv, dtype=dtype_map)

        for c in time_cols:
            self.df[c] = self.df[c].str.replace(r"\.0$", "", regex=True)

    def __len__(self):
        return len(self.df)

    @staticmethod
    def _tok(x) -> str:
        s = str(x)
        if s.endswith(".0"):
            s = s[:-2]
        return s


    def _read_patch(self, path, r, c):
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"Missing raster: {path}")
    
        with rasterio.open(path) as src:
            nodata = src.nodata
            window = Window(col_off=c, row_off=r, width=self.patch, height=self.patch)
            patch = src.read(1, window=window)
    
        if nodata is not None:
            patch = patch.astype(np.float32)
            patch[patch == nodata] = -9999.0
    
        return patch


    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        year = int(row.year)
        r = int(row.patch_row)
        c = int(row.patch_col)

        frames = []
        for t in [self._tok(row.t1), self._tok(row.t2), self._tok(row.t3)]:
            channels = []
            for var in self.variables:
                if var in ["elevation", "slope"]:
                    path = self.data_root / "static" / f"{var}_static_srtm.tif"
                else:
                    path = self.data_root / var / str(year) / f"{var}_{year}_{t}.tif"

                patch = self._read_patch(path, r, c)
                channels.append(patch)

            frame = np.stack(channels, axis=0)
            frames.append(frame)

        X_np = np.stack(frames, axis=0).astype(np.float32)
        X_np[X_np == -9999.0] = 0.0

        t4 = self._tok(row.t4)
        fire_path = self.fire_root / str(year) / f"fire16_{year}_{t4}.tif"
        y_np = self._read_patch(fire_path, r, c).astype(np.float32)
        y_np[y_np == -9999.0] = 0.0

        X = torch.from_numpy(X_np)                 # (T, C, H, W)
        y = torch.from_numpy(y_np)                 # (H, W)

        y = (y > 0).float()

        if not (torch.all(X >= 0.0) and torch.all(X <= 1.0)):
            raise ValueError(f"Input out of [0,1]. min={float(X.min())}, max={float(X.max())}")

        return X, y


# =========================
# MODEL
# =========================
class Simple3DCNN(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv3d(in_channels, 32, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),

            nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),

            nn.Conv3d(64, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
        )
        self.time_pool = nn.AdaptiveAvgPool3d((1, None, None))
        self.head = nn.Conv3d(64, 1, kernel_size=1)

    def forward(self, x):
        x = x.permute(0, 2, 1, 3, 4).contiguous()
        x = self.backbone(x)
        x = self.time_pool(x)
        x = self.head(x)
        return x.squeeze(2)  # (B,1,H,W)


# =========================
# METRICS
# =========================
def dice_iou_from_logits(logits, targets, thr=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs >= thr).float()

    if targets.ndim == 3:
        targets = targets.unsqueeze(1)

    targets = targets.float()
    inter = (preds * targets).sum(dim=(1, 2, 3))
    union = (preds + targets - preds * targets).sum(dim=(1, 2, 3))

    dice = (2 * inter + eps) / (preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps)
    iou = (inter + eps) / (union + eps)

    return dice.mean().item(), iou.mean().item()


def precision_recall_from_logits(logits, targets, thr=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs >= thr).float()

    if targets.ndim == 3:
        targets = targets.unsqueeze(1)

    targets = targets.float()
    tp = (preds * targets).sum(dim=(1, 2, 3))
    fp = (preds * (1 - targets)).sum(dim=(1, 2, 3))
    fn = ((1 - preds) * targets).sum(dim=(1, 2, 3))

    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)

    return precision.mean().item(), recall.mean().item()


# =========================
# LOADERS
# =========================
def make_train_loader():
    train_ds = FireConvLSTMDataset(TRAIN_CSV, DATA_ROOT, FIRE_ROOT, VARS, patch_size=PATCH_SIZE)

    y_patch = train_ds.df["has_fire"].astype(int).to_numpy()
    counts = np.bincount(y_patch, minlength=2)

    w0 = 1.0 / max(counts[0], 1)
    w1 = 1.0 / max(counts[1], 1)
    weights = np.where(y_patch == 1, w1, w0).astype(np.float32)

    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True
    )

    dl_kwargs = dict(
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    if NUM_WORKERS > 0:
        dl_kwargs["persistent_workers"] = True

    dl_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
    )
    
    train_loader = DataLoader(
    train_ds,
    sampler=sampler,
    **dl_kwargs
    )
    return train_ds, train_loader, counts


def make_eval_loader(csv_path):
    ds = FireConvLSTMDataset(csv_path, DATA_ROOT, FIRE_ROOT, VARS, patch_size=PATCH_SIZE)

    dl_kwargs = dict(
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
    )

    loader = DataLoader(ds, **dl_kwargs)

    return ds, loader


# =========================
# POS WEIGHT ESTIMATION
# =========================
@torch.no_grad()
def estimate_pos_weight(train_loader, device, max_batches=200):
    pos = 0.0
    neg = 0.0
    batches = 0

    for _, y in train_loader:
        y = y.to(device, non_blocking=True)
        pos += y.sum().item()
        neg += (y.numel() - y.sum().item())
        batches += 1
        if batches >= max_batches:
            break

    if pos < 1:
        return 1.0, 1.0

    raw = neg / pos
    clipped = float(np.clip(raw, 1.0, 200.0))
    return float(raw), clipped


# =========================
# TRAINING
# =========================
def run_epoch(model, loader, optimizer, criterion, device, train=True, thr=0.5, desc=""):
    model.train() if train else model.eval()

    total_loss = 0.0
    n_batches = 0
    dice_vals, iou_vals, p_vals, r_vals = [], [], [], []

    pbar = tqdm(loader, desc=desc, leave=False)
    for X, y in pbar:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        logits = model(X)
        loss = criterion(logits, y.unsqueeze(1))

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        n_batches += 1

        d, j = dice_iou_from_logits(logits.detach(), y.detach(), thr=thr)
        p, r = precision_recall_from_logits(logits.detach(), y.detach(), thr=thr)

        dice_vals.append(d)
        iou_vals.append(j)
        p_vals.append(p)
        r_vals.append(r)

        pbar.set_postfix({"loss": f"{loss.item():.4f}", "dice": f"{d:.3f}", "iou": f"{j:.3f}"})

    return {
        "loss": total_loss / max(n_batches, 1),
        "dice": float(np.mean(dice_vals)),
        "iou": float(np.mean(iou_vals)),
        "precision": float(np.mean(p_vals)),
        "recall": float(np.mean(r_vals)),
    }


@torch.no_grad()
def threshold_sweep(model, loader, device, thresholds):
    model.eval()
    rows = []
    for thr in thresholds:
        metrics = run_epoch(
            model,
            loader,
            optimizer=None,
            criterion=nn.BCEWithLogitsLoss(),  # placeholder, not used for metrics only
            device=device,
            train=False,
            thr=thr,
            desc=f"VAL sweep thr={thr}"
        )
        rows.append({"thr": thr, **metrics})
    df = pd.DataFrame(rows)
    return df


@torch.no_grad()
def eval_metrics(model, loader, criterion, device, thr=0.5, desc="EVAL"):
    return run_epoch(model, loader, optimizer=None, criterion=criterion, device=device, train=False, thr=thr, desc=desc)


def save_json(path: Path, obj: dict):
    path.write_text(json.dumps(obj, indent=2))


def main():
    # Save config immediately so even if it fails you know what ran
    config = {
        "DATASET_DIR": DATASET_DIR,
        "DATA_ROOT": DATA_ROOT,
        "FIRE_ROOT": FIRE_ROOT,
        "TRAIN_CSV": TRAIN_CSV,
        "VAL_CSV": VAL_CSV,
        "TEST_CSV": TEST_CSV,
        "VARS": VARS,
        "PATCH_SIZE": PATCH_SIZE,
        "BATCH_SIZE": BATCH_SIZE,
        "EPOCHS": EPOCHS,
        "LR": LR,
        "THR_DEFAULT": THR,
        "NUM_WORKERS": NUM_WORKERS,
        "RUN_DIR": str(RUN_DIR),
    }
    save_json(CFG_JSON, config)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    print("Run dir:", RUN_DIR)

    # Data
    train_ds, train_loader, counts = make_train_loader()
    _, val_loader = make_eval_loader(VAL_CSV)
    _, test_loader = make_eval_loader(TEST_CSV)

    print("Train size:", len(train_ds), "class counts [0,1]:", counts, "patch fire ratio:", counts[1] / counts.sum())

    Xb, yb = next(iter(train_loader))
    print("One train batch X:", tuple(Xb.shape), "y:", tuple(yb.shape))
    print("X range:", float(Xb.min()), float(Xb.max()))
    print("y unique:", torch.unique(yb))

    # Model
    model = Simple3DCNN(in_channels=len(VARS)).to(device)

    # Loss weights
    raw_w, pos_weight = estimate_pos_weight(train_loader, device=device, max_batches=200)
    print("Estimated pixel pos_weight raw:", raw_w, "clipped:", pos_weight)

    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    # Init log
    log_rows = []
    log_df = pd.DataFrame(columns=[
        "epoch", "train_loss", "train_dice", "train_iou", "train_precision", "train_recall",
        "val_loss", "val_dice", "val_iou", "val_precision", "val_recall",
        "epoch_time_sec"
    ])
    log_df.to_csv(LOG_CSV, index=False)

    best_val_dice = -1.0

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()

        tr = run_epoch(model, train_loader, optimizer, criterion, device, train=True, thr=THR, desc=f"Epoch {epoch}/{EPOCHS} TRAIN")
        va = run_epoch(model, val_loader, optimizer=None, criterion=criterion, device=device, train=False, thr=THR, desc=f"Epoch {epoch}/{EPOCHS} VAL")

        dt = time.time() - t0

        row = {
            "epoch": epoch,
            "train_loss": tr["loss"], "train_dice": tr["dice"], "train_iou": tr["iou"], "train_precision": tr["precision"], "train_recall": tr["recall"],
            "val_loss": va["loss"], "val_dice": va["dice"], "val_iou": va["iou"], "val_precision": va["precision"], "val_recall": va["recall"],
            "epoch_time_sec": dt
        }
        log_rows.append(row)
        pd.DataFrame(log_rows).to_csv(LOG_CSV, index=False)

        print(
            f"Epoch {epoch:02d} | "
            f"train loss {tr['loss']:.4f} dice {tr['dice']:.4f} iou {tr['iou']:.4f} P {tr['precision']:.4f} R {tr['recall']:.4f} | "
            f"val loss {va['loss']:.4f} dice {va['dice']:.4f} iou {va['iou']:.4f} P {va['precision']:.4f} R {va['recall']:.4f} | "
            f"time {dt:.1f}s"
        )

        # Save last checkpoint each epoch
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "pos_weight": pos_weight,
                "pos_weight_raw": raw_w,
                "thr_default": THR,
                "vars": VARS,
                "patch_size": PATCH_SIZE,
                "config": config
            },
            CKPT_LAST
        )

        # Save best checkpoint
        if va["dice"] > best_val_dice:
            best_val_dice = va["dice"]
            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "pos_weight": pos_weight,
                    "pos_weight_raw": raw_w,
                    "thr_default": THR,
                    "vars": VARS,
                    "patch_size": PATCH_SIZE,
                    "config": config
                },
                CKPT_BEST
            )
            print("Saved best checkpoint:", str(CKPT_BEST))

    print("Training complete. Best val dice:", best_val_dice)

    # Load best and evaluate
    ckpt = torch.load(CKPT_BEST, map_location=device)
    model.load_state_dict(ckpt["model_state"])

    val_metrics = eval_metrics(model, val_loader, criterion, device, thr=THR, desc="VAL final")
    test_metrics = eval_metrics(model, test_loader, criterion, device, thr=THR, desc="TEST thr=default")

    save_json(RUN_DIR / "metrics_val_defaultthr.json", val_metrics)
    save_json(RUN_DIR / "metrics_test_defaultthr.json", test_metrics)

    # Threshold sweep on validation, choose best dice
    sweep_df = []
    model.eval()
    with torch.no_grad():
        rows = []
        for thr in THR_SWEEP:
            m = eval_metrics(model, val_loader, criterion, device, thr=thr, desc=f"VAL thr={thr}")
            rows.append({"thr": thr, **m})
        sweep_df = pd.DataFrame(rows)

    sweep_path = RUN_DIR / "threshold_sweep_val.csv"
    sweep_df.to_csv(sweep_path, index=False)

    best_row = sweep_df.sort_values("dice", ascending=False).iloc[0].to_dict()
    best_thr = float(best_row["thr"])
    save_json(RUN_DIR / "best_threshold.json", {"best_thr": best_thr, "best_row": best_row})

    test_metrics_best = eval_metrics(model, test_loader, criterion, device, thr=best_thr, desc=f"TEST thr={best_thr}")
    save_json(RUN_DIR / "metrics_test_bestthr.json", test_metrics_best)

    print("Default thr test dice:", test_metrics["dice"], "Best thr:", best_thr, "Best thr test dice:", test_metrics_best["dice"])

    # Save sample images
    try:
        import matplotlib.pyplot as plt

        model.eval()
        saved = 0
        with torch.no_grad():
            for X, y in tqdm(test_loader, desc="Saving sample images", leave=False):
                X = X.to(device, non_blocking=True)
                logits = model(X)
                probs = torch.sigmoid(logits).detach().cpu().numpy()  # (B,1,H,W)
                y_np = y.detach().cpu().numpy()                       # (B,H,W)

                for i in range(probs.shape[0]):
                    if saved >= MAX_SAVE_IMAGES:
                        break

                    pred = probs[i, 0]
                    gt = y_np[i]

                    fig = plt.figure(figsize=(9, 3))
                    ax1 = fig.add_subplot(1, 3, 1)
                    ax2 = fig.add_subplot(1, 3, 2)
                    ax3 = fig.add_subplot(1, 3, 3)

                    ax1.imshow(pred, interpolation="nearest")
                    ax1.set_title("Pred prob")
                    ax1.axis("off")

                    ax2.imshow(gt, interpolation="nearest")
                    ax2.set_title("GT mask")
                    ax2.axis("off")

                    ax3.imshow((pred >= best_thr).astype(np.float32), interpolation="nearest")
                    ax3.set_title(f"Pred bin thr={best_thr:.2f}")
                    ax3.axis("off")

                    out_path = SAMPLES_DIR / f"sample_{saved:03d}.png"
                    fig.tight_layout()
                    fig.savefig(out_path, dpi=140)
                    plt.close(fig)

                    saved += 1

                if saved >= MAX_SAVE_IMAGES:
                    break

        print("Saved sample images to:", str(SAMPLES_DIR))
    except Exception as e:
        print("Image saving failed:", str(e))

    # Create a tarball so you can download everything in one file
    tar_path = Path("/kaggle/working") / f"{RUN_DIR.name}.tar.gz"
    os.system(f"tar -czf {tar_path} -C {RUN_DIR.parent} {RUN_DIR.name}")
    print("Saved run archive:", str(tar_path))
    print("Done. Download the tar.gz from Kaggle output files.")

if __name__ == "__main__":
    try:
        main()
    except Exception:
        # log full traceback so you can debug after the run
        tb = traceback.format_exc()
        ERR_LOG.write_text(tb)
        print("Run failed. Traceback saved to:", str(ERR_LOG))
        raise


Device: cuda
Run dir: /kaggle/working/runs/3dcnn_20260110_025955
Train size: 21117 class counts [0,1]: [14706  6411] patch fire ratio: 0.30359426054837335


/tmp/ipykernel_55/463477349.py:57: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_ID = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


One train batch X: (32, 3, 7, 32, 32) y: (32, 32, 32)
X range: 0.0 0.9713746309280396
y unique: tensor([0., 1.])
Estimated pixel pos_weight raw: 286.4764223362723 clipped: 200.0


Epoch 1/6 TRAIN:   0%|          | 0/660 [00:13<?, ?it/s]

Epoch 1/6 VAL:   0%|          | 0/115 [00:00<?, ?it/s]

...

VAL thr=0.9:   0%|          | 0/115 [00:00<?, ?it/s]

TEST thr=0.9:   0%|          | 0/116 [00:00<?, ?it/s]

Default thr test dice: 0.5030536806042274 Best thr: 0.9 Best thr test dice: 0.7159686355845124


Saving sample images:   0%|          | 0/116 [00:00<?, ?it/s]

Saved sample images to: /kaggle/working/runs/3dcnn_20260110_025955/pred_samples
Saved run archive: /kaggle/working/3dcnn_20260110_025955.tar.gz
Done. Download the tar.gz from Kaggle output files.


In [5]:
import torch
print(torch.cuda.is_available())  # Should print: True
print(torch.cuda.get_device_name(0))  # Should print: Tesla T4

True
Tesla T4


In [6]:
!zip -r all_files.zip /kaggle/working

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/3dcnn_20260110_025955.tar.gz (deflated 0%)
  adding: kaggle/working/runs/ (stored 0%)
  adding: kaggle/working/runs/3dcnn_20260110_025955/ (stored 0%)
  adding: kaggle/working/runs/3dcnn_20260110_025955/threshold_sweep_val.csv (deflated 57%)
  adding: kaggle/working/runs/3dcnn_20260110_025955/last.pt (deflated 8%)
  adding: kaggle/working/runs/3dcnn_20260110_025955/metrics_test_defaultthr.json (deflated 28%)
  adding: kaggle/working/runs/3dcnn_20260110_025955/metrics_test_bestthr.json (deflated 30%)
  adding: kaggle/working/runs/3dcnn_20260110_025955/pred_samples/ (stored 0%)
  adding: kaggle/working/runs/3dcnn_20260110_025955/pred_samples/sample_022.png (deflated 24%)
  adding: kaggle/working/runs/3dcnn_20260110_025955/pred_samples/sample_020.png (deflated 24%)
  adding: kaggle/working/runs/3dcnn_20260110_025955/pred_samples/sample_001.png (deflated 25%)
  adding: kaggle/working/runs/3dcnn_20260110_025955/pred_samples/samp